# 09 - DataLoader (AI Infra 视角)

本节从 **工程实现** 角度理解 LLM 数据加载：
- Parquet 存储格式
- 分布式数据分片
- 异步加载与预取
- 断点续训

In [ ]:
import torch
from collections import deque

## 1. LLM 训练数据流 (30秒版)

```
Parquet 文件 (TB级)
    ↓ 分片到各 GPU
Row Group
    ↓ 读取文本
List[str]
    ↓ Tokenize
List[int]
    ↓ 拼接 + BOS
Token Buffer (deque)
    ↓ 取 B*T+1 个
(inputs, targets)
    ↓ pin_memory + non_blocking
GPU Tensor
```

## 2. Parquet 格式

**为什么用 Parquet？**
- 列式存储，压缩率高
- 支持 Row Group，可分块读取
- 适合大规模数据

```
shard_00000.parquet
├── Row Group 0 (1024 rows)
│   └── column 'text': ["doc1...", "doc2...", ...]
├── Row Group 1 (1024 rows)
│   └── column 'text': [...]
└── ...

shard_00001.parquet
├── Row Group 0
└── ...

(共 1823 个 shard 文件)
```

In [1]:
# Parquet 文件大小建议
print("Parquet 文件大小建议:")
print("  单个文件: 128MB - 1GB (推荐 256-512MB)")
print("  Row Group: 128MB (Parquet 默认)")
print("")
print("太小: 元数据开销大，调度开销大")
print("太大: 并行度受限，故障恢复慢")

Parquet 文件大小建议:
  单个文件: 128MB - 1GB (推荐 256-512MB)
  Row Group: 128MB (Parquet 默认)

太小: 元数据开销大，调度开销大
太大: 并行度受限，故障恢复慢


## 3. 分布式数据分片 (DDP)

**核心**: 每个 GPU 读取不同的数据

```
Row Groups: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, ...]

World Size = 4 (4 个 GPU)

Rank 0: [0, 4, 8,  ...]  从 0 开始，步长 4
Rank 1: [1, 5, 9,  ...]  从 1 开始，步长 4  
Rank 2: [2, 6, 10, ...]  从 2 开始，步长 4
Rank 3: [3, 7, 11, ...]  从 3 开始，步长 4
```

In [ ]:
# 分布式数据加载
def get_row_groups_for_rank(total_rg, rank, world_size):
    """计算当前 rank 应该读取的 row groups"""
    return list(range(rank, total_rg, world_size))

# 示例
total_rg = 16
world_size = 4

for rank in range(world_size):
    rgs = get_row_groups_for_rank(total_rg, rank, world_size)
    print(f"Rank {rank}: {rgs}")

## 4. Token Buffer 机制

**问题**: 文档长度不一，需要凑成固定大小的 batch

**解决**: 用 deque 作为 buffer，累积 token

In [ ]:
class TokenBuffer:
    """
    Token 缓冲区
    - 累积 token 直到足够一个 batch
    - 文档之间用 BOS token 分隔
    """
    def __init__(self, bos_token=0):
        self.buffer = deque()
        self.bos_token = bos_token
    
    def add_document(self, tokens):
        """添加一个文档的 token"""
        self.buffer.append(self.bos_token)
        self.buffer.extend(tokens)
    
    def get_batch(self, batch_size, seq_len):
        """取出 B*T+1 个 token 构造 batch"""
        needed = batch_size * seq_len + 1
        
        if len(self.buffer) < needed:
            return None  # 不够
        
        tokens = [self.buffer.popleft() for _ in range(needed)]
        
        # 构造 inputs 和 targets
        inputs = torch.tensor(tokens[:-1]).view(batch_size, seq_len)
        targets = torch.tensor(tokens[1:]).view(batch_size, seq_len)
        
        return inputs, targets

# 测试
buffer = TokenBuffer()
buffer.add_document([1, 2, 3, 4, 5])
buffer.add_document([6, 7, 8, 9, 10])
print(f"Buffer 内容: {list(buffer.buffer)[:15]}...")
print(f"Buffer 长度: {len(buffer.buffer)}")

## 5. 异步加载优化 (重要!)

**目标**: GPU 计算时，CPU 准备下一个 batch

```
时间线:

GPU:  [  训练 batch 0  ][  训练 batch 1  ][  训练 batch 2  ]
CPU:  [准备 batch 1][准备 batch 2][准备 batch 3]
       ↑ 重叠执行
```

### 关键技术

1. **pin_memory**: 锁页内存，加速 CPU→GPU 传输
2. **non_blocking**: 异步传输，不阻塞 CPU
3. **prefetch**: 提前准备下一个 batch

In [ ]:
# 异步加载示例
async_loading_code = '''
# 1. 用锁页内存创建 tensor
tokens = torch.tensor(data, dtype=torch.long, pin_memory=True)

# 2. 异步传输到 GPU
inputs = inputs_cpu.to(device="cuda", non_blocking=True)
targets = targets_cpu.to(device="cuda", non_blocking=True)

# 3. 训练循环中 prefetch
next_batch = dataloader.get_batch()  # CPU 准备下一个

for step in range(num_steps):
    x, y = next_batch  # 当前 batch
    next_batch = dataloader.get_batch()  # GPU 算的时候，CPU 准备下一个
    
    loss = model(x, y)
    loss.backward()
    optimizer.step()
'''
print(async_loading_code)

## 6. 断点续训 (Resume)

**保存状态**:
- 当前 Parquet 文件索引
- 当前 Row Group 索引
- Token buffer 内容

```python
resume_state = {
    "pq_idx": 5,     # 当前 parquet 文件
    "rg_idx": 128,   # 当前 row group
    "buffer": [...], # buffer 内容 (可选)
}
```

## 7. 面试常见问题

### Q1: 为什么用 Parquet 而不是 JSON/TXT?

**答**:
- 列式存储，压缩率高 (通常 5-10x)
- 支持 Row Group，可分块读取，不需要全部加载
- Schema 自描述，支持复杂类型

---

### Q2: 如何保证 DDP 各 GPU 数据不重复?

**答**:
- 按 Row Group 分片，每个 GPU 从不同起点开始
- `rank_i` 读取 `[i, i+world_size, i+2*world_size, ...]`
- 或用 DistributedSampler

---

### Q3: pin_memory 是什么?

**答**:
- 锁页内存，不会被 swap 到磁盘
- GPU 可以直接 DMA 访问，省去一次拷贝
- CPU→GPU 传输速度提升 2-3x

---

### Q4: 为什么文档之间要加 BOS token?

**答**:
- 标记文档边界
- 防止跨文档学习错误的关联
- 推理时作为起始 token

---

### Q5: DataLoader 的性能瓶颈通常在哪?

**答**:
1. **Tokenization**: CPU 密集，考虑用 Rust 实现
2. **IO**: 磁盘读取慢，用 NVMe SSD
3. **CPU→GPU 传输**: 用 pin_memory + non_blocking

---

### Q6: 如何处理超长文档?

**答**:
- 截断到 max_seq_len
- 或分块处理，每块加 BOS
- 保持文档完整性 vs 训练效率的权衡

## 8. 总结速查表

| 主题 | 要点 |
|------|------|
| **Parquet** | 列式存储，Row Group 分块，压缩率高 |
| **DDP 分片** | 每个 rank 从不同 row group 开始，步长=world_size |
| **Token Buffer** | deque 累积 token，凑够 B*T+1 再 yield |
| **异步加载** | pin_memory + non_blocking + prefetch |
| **文档分隔** | 每个文档前加 BOS token |
| **断点续训** | 保存 (pq_idx, rg_idx) |

### 数据流公式

```
inputs  = tokens[0 : B*T].view(B, T)
targets = tokens[1 : B*T+1].view(B, T)
# targets 就是 inputs 右移一位
```